In [0]:
data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")]
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")]

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))

In [0]:
df_1.selectExpr("string_time" ,
                '''to_timestamp(string_time, "yyyy-MM-dd'T'HH:mm:ss.SSSS") as valid_timestamp''').display()

from pyspark.sql.functions import col , to_timestamp
df_1.select("string_time" , 
            to_timestamp(col("string_time") , "yyyy-MM-dd'T'HH:mm:ss.SSSS").alias("valid_time")).display()

In [0]:
spark.sql('''select current_timestamp ''').show(truncate = False)

In [0]:
spark.sql(''' select to_timestamp('18-05-2022 10:30:30.0000' , 'dd-MM-yyyy HH:mm:ss.SSSS')  ''').display()

In [0]:
spark.conf.get('spark.sql.session.timeZone')

In [0]:
spark.conf.set('spark.sql.session.timeZone' , 'IST')

In [0]:
spark.sql(''' select to_timestamp('18-05-2022 10:30:30.0000' , 'dd-MM-yyyy HH:mm:ss.SSSS')  ''').display()

In [0]:
spark.conf.set('spark.sql.session.timeZone' , 'Etc/UTC')

In [0]:
schema = 'component string , event_time string , reading string'

df = spark.read.format("csv")\
                .option("header" , "true")\
                    .schema(schema)\
                        .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-no-tz.csv')
df.display()

In [0]:
from pyspark.sql.functions import to_timestamp_ntz,col,lit

df1 = df.withColumn("event_time_valide_ntz" , to_timestamp_ntz('event_time' , lit('dd-MM-yyyy HH:mm:ss.SSS')))
df1.display()
df1.printSchema()

In [0]:
# to convert timestamp ist to utu
# check the below columns event_time_valide_ntz and event_time_tz
#convert_timezone(sourceTz , targetTz , sourceTs)
from pyspark.sql.functions import convert_timezone

df2 = df1.withColumn('event_time_tz' , convert_timezone(lit('IST') , lit('UTC') , 'event_time_valide_ntz'))
df2.display()
df2.printSchema()

In [0]:
# to convert timestamp_ntz to timestamp
#stz = session timezone

df3 = df1.withColumn('event_time_tz' , to_timestamp(convert_timezone(lit('IST') , lit('stz') , 'event_time_valide_ntz')))
df3.printSchema()

In [0]:
df = spark.read.format('csv')\
            .option('header' , 'true')\
                .schema(schema)\
                .load(path = '/Volumes/dev/spark_db/datasets/spark_programming/data/machine-events-with-tz.csv')
df.printSchema()
df.display()

In [0]:
df.withColumn('event_time_tz' , to_timestamp('event_time' , 'dd-MM-yyyy HH:mm:ss.SSSZ')).display()
# convert timezone to Ssession time zone